In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/clinical_outcome/bulk_counts_clinical_outcome.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['ClinicalOutcome'] = adata.obs['ClinicalOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
adata.obs['ModelOutcome2'] = adata.obs['ModelOutcome2'].map({'Average':'Good', 'Poor':'Poor'})
adata.obs['ModelOutcome1'] = adata.obs['ModelOutcome1'].map({'Average':'Good', 'Poor':'Poor'})
# Cosmetic relabelling only: every raw Site value keeps its own category.
# 'BOT' is deliberately NOT folded into 'Base of Tongue' - they stay separate sites.
adata.obs['Site'] = adata.obs['Site'].astype(str).replace({
    'Baseoftongue':   'Base of Tongue',
    'BuccalMucosa':   'Buccal Mucosa',
    'Maxillarysinus': 'Maxillary Sinus',
    'FloorofMouth':   'Floor of Mouth',
})

In [ ]:
adata.obs.rename(columns={
    'ClinicalOutcome':'Clinical Outcome',
    'DEGOutcome':'DEG Outcome',
    'ModelOutcome1':'ModelOutcome1',
    'ModelOutcome2':'Model Outcome'
}, inplace = True)

In [ ]:
# normalization and log transformation of expression values.
adata_tmp = adata.copy()
sc.pp.normalize_total(adata_tmp, target_sum=1e4)  # library-size normalize
sc.pp.log1p(adata_tmp)

In [ ]:
adata_tmp.var_names = adata_tmp.var['gene_label']

In [ ]:
# expression matrix: samples x genes
expr = pd.DataFrame(
    adata_tmp.X,
    index=adata_tmp.obs_names,
    columns=adata_tmp.var_names
)

In [ ]:
expr.head()

In [ ]:
# ---------------------------------------
# 2) Pick genes to show (DE-driven subset)
#    (padj<0.05, highest |log2FC|, and optionally baseMean filter)
# ---------------------------------------
de = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# Optional low-expression filter to avoid noisy genes
BASEMEAN_MIN = 10 # adjust as you like or set to 0 to disable
if "baseMean" in de.columns:
    de = de[de["baseMean"] >= BASEMEAN_MIN]

# gene_label falls back to the Ensembl ID when the gene has no symbol (18,355 of 60,279
# genes here; gene_label == gene_id for exactly those). Bare ENSG rows say nothing to a
# reader, so drop them BEFORE ranking - that way the figure still gets a full N up and
# N down rather than losing rows.
de = de[de["gene_label"].astype(str) != de["gene_id"].astype(str)]

# choose top up/down (e.g., 25 each)
N = 15
up = de[(de["padj"] < 0.05) & (de["log2FoldChange"] > 0)].nsmallest(N, "padj")
dn = de[(de["padj"] < 0.05) & (de["log2FoldChange"] < 0)].nsmallest(N, "padj")

def names(df):
    if "gene_label" in df.columns:
        sym = df["gene_label"].astype(str)
        # Replace 'nan' with index
        return sym.where(sym != 'nan', df.index).astype(str)
    return df.index.astype(str)

genes = pd.Index(names(up).tolist() + names(dn).tolist())
genes = genes.intersection(expr.columns)  # keep only those present in expression

print(f"Found {len(up)} upregulated genes and {len(dn)} downregulated genes")
print(f"Total genes for heatmap (before adding genes of interest): {len(genes)}")

# ---------------------------------------
# Add genes of interest
# ---------------------------------------
genes_of_interest = [
    'MYB',
    'MYBL1',          
    'MYC',          
    'NOTCH1',
    'NFIB',
    'KIT',
    'EGFR',
    'TP53',
    'PTEN',
    'SOX4',
    'CDK6'
]

# Add genes of interest to the selection
genes_to_add = pd.Index(genes_of_interest).intersection(expr.columns)
genes = genes.union(genes_to_add)

print(f"Added {len(genes_to_add)} genes of interest: {', '.join(genes_to_add)}")
if len(genes_to_add) < len(genes_of_interest):
    missing = set(genes_of_interest) - set(genes_to_add)
    print(f"Warning: {len(missing)} genes not found in expression data: {', '.join(missing)}")
print(f"Total genes for heatmap: {len(genes)}")

# Identify which genes of interest are significantly DE
de_full = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()
sig_de_genes = set(names(de_full[(de_full["padj"] < 0.05)]))
non_sig_genes_of_interest = [g for g in genes_of_interest if g not in sig_de_genes and g in expr.columns]

print(f"Genes of interest that are NOT significantly DE: {', '.join(non_sig_genes_of_interest) if non_sig_genes_of_interest else 'None'}")

# ---------------------------------------
# 3) Build z-scored matrix (per gene)
# ---------------------------------------
mat = expr[genes]  # samples x selected genes
mat_z = (mat - mat.mean(axis=0)) / mat.std(axis=0)

# Drop any genes with NaN or infinite values after z-scoring
mat_z = mat_z.loc[:, ~mat_z.isnull().any(axis=0)]
mat_z = mat_z.loc[:, ~np.isinf(mat_z).any(axis=0)]

print(f"After removing NaN/Inf genes: {mat_z.shape[1]} genes remaining")

# Order samples by metadata (columns are not re-clustered, so this fixes display order)
# Clinical Outcome is the primary key so Good/Poor form contiguous blocks
# ("Good" < "Poor" alphabetically -> Good ends up on the left, Poor on the right)
order_cols = [c for c in ["Clinical Outcome", "Site"] if c in adata.obs.columns]
if order_cols:
    sample_order = adata.obs.sort_values(order_cols).index
    mat_z = mat_z.loc[sample_order]


# ---------------------------------------
# 4) Column annotations (sample bars) - Multiple annotations
# ---------------------------------------
# Define which columns to show as annotation bars
annotation_cols = ['Clinical Outcome', 'Site']

# Create color mappings for each annotation
col_colors_dict = {}

# One fixed colour per site - never cycled, so two different sites can never end up
# sharing a colour (the old 7-colour list was indexed with `i % len(...)`, which repainted
# sites 8+ with colours already in use). Chosen by max-min OKLab separation: passes the
# categorical checks all-pairs on a light surface (worst CVD dE 8.7 >= 8 target; worst
# normal-vision dE 15.9 >= 15 floor) while staying off the Poor (#E63946) / Good (#2c9d2a)
# outcome hues.
SITE_COLORS = {
    'BOT':             '#dc829f',
    'Base of Tongue':  '#cc6004',
    'Buccal Mucosa':   '#804810',
    'Floor of Mouth':  '#95af1f',
    'Larynx':          '#22876b',
    'Mandible':        '#46aedb',
    'Maxilla':         '#35578f',
    'Maxillary Sinus': '#2e20f0',
    'Parotid':         '#716ff1',
    'Submandibular':   '#c740c0',
    'Tongue':          '#9d0367',
}
assert len(set(SITE_COLORS.values())) == len(SITE_COLORS), "duplicate colour in SITE_COLORS"

for col in annotation_cols:
    if col in adata.obs.columns:
        unique_vals = adata.obs[col].dropna().astype(str).unique()

        if col == 'Clinical Outcome':
            # Fixed brand colors regardless of category order
            fixed_lut = {'Poor': '#E63946', 'Good': '#2c9d2a'}
            lut = {val: fixed_lut.get(val, '#B0B0B0') for val in unique_vals}
        elif col == 'Site':
            # Fail loudly on an unknown site rather than silently reusing a colour.
            missing_sites = sorted(set(unique_vals) - set(SITE_COLORS))
            if missing_sites:
                raise KeyError(f"No colour assigned for site(s): {missing_sites} - add them to SITE_COLORS")
            lut = {val: SITE_COLORS[val] for val in unique_vals}
        else:
            palette = sns.color_palette("Set2", n_colors=len(unique_vals))
            lut = dict(zip(unique_vals, palette))

        col_colors_dict[col] = adata.obs.loc[mat_z.index, col].astype(str).map(lut)

# Convert to DataFrame
col_colors = pd.DataFrame(col_colors_dict) if col_colors_dict else None

In [ ]:
# ---------------------------------------
# Alternative heatmap: fully custom matplotlib layout (no sns.clustermap).
#
# The clustermap-based cell above kept fighting itself: cbar_pos/dendrogram_ratio
# are coordinates into clustermap's own auto-computed GridSpec, so nudging one
# shifts where everything else lands, and plt.tight_layout() then re-fights the
# manually-placed colorbar/legend on top of that. Here every panel gets its own
# fixed GridSpec cell instead, so nothing can overlap regardless of font sizes
# or category counts. Reuses mat_z, adata, annotation_cols, col_colors_dict,
# genes_of_interest, non_sig_genes_of_interest computed in the cell above.
# ---------------------------------------
import os
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.patches import Patch
from matplotlib.colors import to_rgb

# --- Figure typography: Arial Narrow at 9 pt throughout -------------------------
# 'Arial Narrow' is proprietary and is NOT installed on this machine; 'Liberation Sans
# Narrow' is its metric-compatible substitute and is what actually renders here. Arial
# Narrow is listed first so any machine that does have it picks up the real font.
# svg.fonttype='none' keeps text as text in the SVG (editable in Illustrator, and it
# resolves the font at open time) instead of matplotlib's default outlined paths.
FONT_STACK = ['Arial Narrow', 'Liberation Sans Narrow', 'Nimbus Sans Narrow', 'DejaVu Sans']
FONT_SIZE = 9
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': FONT_STACK,
    'font.size': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'legend.title_fontsize': FONT_SIZE,
    'figure.titlesize': FONT_SIZE,
    'svg.fonttype': 'none',
})

sample_names = mat_z.index.tolist()

fig = plt.figure(figsize=(14, 16))
gs = fig.add_gridspec(
    nrows=3, ncols=3,
    width_ratios=[0.4, 8, 2.4],      # [colorbar column, heatmap, legend margin]
    height_ratios=[0.5, 0.6, 10],    # [colorbar row, annotation-strip row, heatmap row]
    hspace=0.04, wspace=0.04,
)

# Colorbar gets its own narrow, dedicated column so its shape (tall/narrow) stays
# fixed regardless of the heatmap column's width - sharing a column with the
# heatmap previously stretched it into a short, wide bar.
ax_cbar = fig.add_subplot(gs[0, 0])
ax_annot = fig.add_subplot(gs[1, 1])
ax_heat = fig.add_subplot(gs[2, 1])

# 1) Cluster genes for ordering only - no dendrogram is drawn (it was overlapping
# the gene name labels, which matplotlib places on the left of ax_heat by default)
row_linkage = linkage(mat_z.T.values, method='average', metric='euclidean')
dendro_data = dendrogram(row_linkage, no_plot=True)

row_order = dendro_data['leaves']
genes_ordered = mat_z.columns[row_order]
heat_data = mat_z[genes_ordered].T.values  # genes x samples, in clustered gene order

# 2) Annotation strips (Clinical Outcome + Site) above the heatmap only
annot_rgb = np.array([
    [to_rgb(c) for c in col_colors_dict[col].loc[sample_names]]
    for col in annotation_cols
])
ax_annot.imshow(annot_rgb, aspect='auto', interpolation='nearest')
ax_annot.set_yticks(range(len(annotation_cols)))
ax_annot.set_yticklabels(annotation_cols, fontsize=FONT_SIZE)
ax_annot.set_xticks([])
for spine in ax_annot.spines.values():
    spine.set_visible(False)

# 3) Main heatmap
im = ax_heat.imshow(heat_data, cmap="RdBu_r", vmin=-3, vmax=3, aspect='auto')
ax_heat.set_xticks(range(len(sample_names)))
ax_heat.set_xticklabels(sample_names, rotation=90, fontsize=FONT_SIZE)
ax_heat.set_xlabel("Samples", fontsize=FONT_SIZE)
ax_heat.set_ylabel("Genes (Z-score)", fontsize=FONT_SIZE)

# Genes of interest are marked by bold + larger type only - no ' *' suffix (the meaning
# of the bolding is explained in the figure caption).
current_labels = list(genes_ordered)
ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(current_labels, fontsize=FONT_SIZE)
for tick_label, gene_name in zip(ax_heat.get_yticklabels(), current_labels):
    if gene_name in genes_of_interest:
        tick_label.set_weight('bold')

# 4) Colorbar - its own fixed, dedicated axis, so its size is predictable
fig.colorbar(im, cax=ax_cbar, label='Z-score')

# 5) Legends - one block per category, reserved legend column, can't overlap the heatmap
legend_y = 0.85
for title in annotation_cols:
    unique_vals = adata.obs.loc[sample_names, title].astype(str).unique()
    handles = [
        Patch(
            facecolor=col_colors_dict[title][adata.obs.loc[sample_names, title].astype(str) == val].iloc[0],
            edgecolor='black', linewidth=0.5
        )
        for val in unique_vals
    ]
    leg = fig.legend(
        handles=handles,
        labels=list(unique_vals),
        title=title,
        loc='upper left',
        bbox_to_anchor=(0.83, legend_y),
        fontsize=FONT_SIZE,
        title_fontsize=FONT_SIZE,
        frameon=True,
    )
    fig.add_artist(leg)  # keep this legend; otherwise the next fig.legend() call replaces it
    legend_y -= 0.25

# No plt.tight_layout() here - the GridSpec ratios already fix the spacing.
fig.subplots_adjust(left=0.05, right=0.98, top=0.98, bottom=0.05)

# Save as SVG
output_dir = f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome'
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, 'heatmap_up_down.svg'), format='svg', bbox_inches='tight')

plt.show()
plt.close()

In [ ]:
list(dn['gene_name'])

In [ ]:
# # Extract top 30 upregulated and top 30 downregulated genes
# de_genes = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# # Optional: apply baseMean filter
# BASEMEAN_MIN = 10
# if "baseMean" in de_genes.columns:
#     de_genes = de_genes[de_genes["baseMean"] >= BASEMEAN_MIN]

# # Get top 30 upregulated genes (positive log2FC, lowest padj)
# top30_up = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] > 0)].nsmallest(30, "padj")

# # Get top 30 downregulated genes (negative log2FC, lowest padj)
# top30_down = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] < 0)].nsmallest(30, "padj")

# # Display upregulated genes
# print("=" * 80)
# print("TOP 30 UPREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_up)} upregulated genes\n")
# up_display = top30_up[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# up_display["Symbol"] = up_display["Symbol"].astype(str)
# print(up_display.to_string())

# # Display downregulated genes
# print("\n" + "=" * 80)
# print("TOP 30 DOWNREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_down)} downregulated genes\n")
# down_display = top30_down[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# down_display["Symbol"] = down_display["Symbol"].astype(str)
# print(down_display.to_string())

# # Optional: Save to CSV files
# top30_up.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_upregulated_genes.csv')
# top30_down.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_downregulated_genes.csv')
# print("\n" + "=" * 80)
# print("Files saved:")
# print("  - top30_upregulated_genes.csv")
# print("  - top30_downregulated_genes.csv")
# print("=" * 80)